# 27. Communication Scheduling Optimization | 通信调度优化

**难度：** Hard | **环境：** CPU-first | **标签：** `并行通信`, `通信优化`, `Overlap` | **目标人群：** 通信机制入门者

---

## 本节导读

本节从通信量、带宽、延迟和同步点出发，理解消息组织与计算重叠如何影响 step time。

你将把一次通信拆成数据量、带宽、启动延迟和同步次数，再观察消息组织、Collective 选择与计算重叠如何改变训练步的关键路径。

**关键词：** `overlap`, `all-reduce`, `all-to-all`

![本节概念关系](../docs/public/01_Hardware_Math_and_Systems/27_communication_schedule_map.svg)



## 前置阅读

**导语：** 先把通信拓扑、并行策略和 NCCL 同步原语对应到一次训练 step，再分析通信放在什么位置、以什么粒度发送，以及哪些计算可以与它重叠。

- [05. Communication Topologies | 通信拓扑与分布式基石](./05_Communication_Topologies.ipynb)
- [20. NCCL and AllReduce Basics | NCCL 与 AllReduce 基础](./20_NCCL_and_AllReduce_Basics.ipynb)
- [26. Parallel Strategy Decision Framework | 并行策略决策框架](./26_Parallel_Strategy_Decision_Framework.ipynb)


## Q1：通信成本如何由数据量、带宽、延迟和同步共同决定？

<details>
<summary>点击展开查看解析</summary>

通信是否挡住计算，先由**位置**决定，再由**带宽**决定。
估算时先拆开数据量、链路带宽、单次启动延迟和同步次数：数据量决定传输下限，带宽决定传输时间，延迟和同步决定小消息或高频通信的额外成本。

如果通信点放在关键路径上，它就会直接暴露成停顿；如果通信能落在计算间隙里，同样的带宽条件下，体感就会完全不同。

所以这一页真正关心的不是“通信有没有”，而是：
- 通信点是不是太密
- 每个通信点是不是都落在关键路径上
- 计算间隙能不能把通信藏住

换句话说，调度决定了通信是否成为瓶颈的可见部分，带宽只是决定它有多重。
</details>


### Q1小验证

拆开数据量、带宽、延迟和同步次数的影响。

In [ ]:
def schedule_cost(comm_points, compute_blocks, overlap_ratio):
    # 调度成本 = 暴露出来的通信点 + 计算块切得太碎带来的额外压力 - overlap 带来的缓冲。
    exposed_points = comm_points * (1 - overlap_ratio)
    fragmentation = max(0, compute_blocks - comm_points)
    pressure = exposed_points * 3 + fragmentation * 0.5
    relief = overlap_ratio * 4
    return {
        'exposed_points': round(exposed_points, 2),
        'fragmentation': fragmentation,
        'schedule_pressure': round(pressure - relief, 2),
    }

for case in [(4, 8, 0.2), (2, 8, 0.6), (6, 4, 0.1)]:
    print(case, '->', schedule_cost(*case))
print('the lower the exposed communication points, the easier the schedule')


## Q2：为什么把小通信合并成大通信通常更稳？

<details>
<summary>点击展开查看解析</summary>

“小通信合并成大通信”真正省下来的，不只是字节数，而是**启动次数、同步次数和调度碎片**。

可以把这件事拆成三层：
- **launch 开销**：每次发起通信都要付固定代价
- **带宽利用**：更大的消息通常更容易吃满链路
- **流水线粒度**：合并太大又会挤压 overlap 空间

所以合并的目标不是“越大越好”，而是“把碎片变少，同时不把流水线切得太粗”。

这也是为什么通信优化经常不是先谈算法，而是先谈消息怎么排、在哪里合、合到什么粒度。
</details>


### Q2小验证

比较合并消息后的启动开销与 overlap 空间。

In [ ]:
def merge_tradeoff(num_small, merged_size_mb, bw_gbps, launch_cost=1.5):
    # 合并消息的收益 = 少发起几次 + 更稳定的带宽利用；代价 = 粒度变粗。
    launches_saved = max(num_small - 1, 0)
    launch_gain = launches_saved * launch_cost
    transfer_cost = merged_size_mb * 8 / bw_gbps
    granularity_penalty = max(merged_size_mb / 128 - 1, 0)
    return {
        'launch_gain': round(launch_gain, 2),
        'transfer_cost': round(transfer_cost, 2),
        'granularity_penalty': round(granularity_penalty, 2),
        'merge_score': round(launch_gain - transfer_cost - granularity_penalty, 2),
    }

for case in [(8, 64, 900), (8, 256, 900), (16, 128, 64)]:
    print(case, '->', merge_tradeoff(*case))
print('merge helps only when fewer launches are worth more than the larger chunk cost')


## Q3：为什么 All-Reduce / All-to-All 的优化目标不同？

<details>
<summary>点击展开查看解析</summary>

这两类通信的瓶颈不一样：

- **All-Reduce** 的核心是把同步代价压进计算之外，重点看是否能和反向传播重叠
- **All-to-All** 的核心是把路由和分发稳定下来，重点看 token 是否会形成局部拥塞

因此，优化目标也不同：
- All-Reduce 更关心**同步压力**和关键路径暴露
- All-to-All 更关心**路由压力**和设备间负载均衡

如果把两者都当成“只是搬数据”，就会错过真正的优化点。
</details>


### Q3小验证

区分 All-Reduce 的同步压力和 All-to-All 的路由压力。

In [ ]:
def comm_goal(kind, sync_pressure, routing_pressure):
    # 不同通信模式的优化目标不同：一个偏同步，一个偏路由。
    if kind == 'allreduce':
        score = sync_pressure * 2 - routing_pressure
        bottleneck = 'sync'
    elif kind == 'alltoall':
        score = routing_pressure * 2 - sync_pressure
        bottleneck = 'routing'
    else:
        score = 0
        bottleneck = 'unknown'
    return {'bottleneck': bottleneck, 'score': score}

for case in [('allreduce', 3, 1), ('alltoall', 1, 3), ('allreduce', 1, 3)]:
    print(case, '->', comm_goal(*case))
print('allreduce and all-to-all should be optimized against different bottlenecks')


## Q4：怎样把通信排进计算间隙？

<details>
<summary>点击展开查看解析</summary>

通信真正能藏进去，靠的不是“把链路弄快一点”，而是把它安排到计算间隙里。

常见手段有三类：
- **移动同步点**：把通信放到更自然的边界，而不是硬塞进关键路径
- **拆分大消息**：让部分通信更早或更晚发生，减少单次暴露
- **利用并发执行**：让独立的 kernel / stream / 传输彼此错开

```mermaid
flowchart LR
    A[Compute A] --> B[Gap]
    B --> C[Compute B]
    A -. communication .-> B
    B -. communication .-> C
    subgraph keypath[Critical Path]
    A --> B --> C
    end
```

所以，真正有效的优化往往不是“单次通信更快”，而是“通信尽量不出现在关键路径上”。
</details>


### Q4小验证

观察通信进入计算间隙后，关键路径成本如何变化。

In [ ]:
def overlap_window(compute_ms, comm_ms, gap_ms):
    # 只有当通信能塞进计算间隙时，overlap 才真正成立。
    hidden = min(comm_ms, gap_ms)
    exposed = max(comm_ms - gap_ms, 0)
    overlap_ratio = hidden / comm_ms if comm_ms else 0
    critical_path = compute_ms + exposed
    return {
        'hidden_comm_ms': round(hidden, 2),
        'exposed_comm_ms': round(exposed, 2),
        'overlap_ratio': round(overlap_ratio, 2),
        'critical_path_ms': round(critical_path, 2),
    }

for case in [(40, 12, 2), (40, 12, 8), (10, 16, 4)]:
    print(case, '->', overlap_window(*case))
print('effective overlap depends on whether the gap can hide the communication')


## 相关阅读

**导语：** 如果还想把通信优化和实现细节连起来，可以接着看异步调度、容错和高级 stream 调度。

- [17. CUDA Stream and Asynchrony | CUDA Stream 与异步执行](./17_CUDA_Stream_and_Asynchrony.ipynb)
- [28. Fault Tolerance and Checkpointing | 容错与检查点](./28_Fault_Tolerance_and_Checkpointing.ipynb)
- [29. CUDA Stream Advanced Scheduling | CUDA Stream 高级调度](./29_CUDA_Stream_Advanced_Scheduling.ipynb)
- [NCCL Documentation | NVIDIA Collective Communications Library](https://docs.nvidia.com/deeplearning/nccl/user-guide/docs/)
- [nccl-tests | NCCL 性能测试工具](https://github.com/NVIDIA/nccl-tests)
---